- 1. Create bronze table as Bronze_Orders
- 2. Ingest data incrementally into the bronze table from the cloud files 
- 3. Add file_path and ingest date
- 4. Create silver clean orders table (delibritaly or via funtion)
- 5. Apply quality checks
- 5. Load data from bronze to silver clean
- 6. Apply transformtions between silver clean and silver tables order_timestamp
- 7. Silver_Orders to include all the orders with the items arry exploded (i.e, there will be one record per each item in the order). 

![Process Orders Data.png](./Process Orders Data.png "Process Orders Data.png")

In [0]:
import dlt
import pyspark.sql.functions as F
from pyspark.sql.types import (
    ArrayType,
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

In [0]:
@dlt.table(
    name = "bronze_orders",
    table_properties = {'quality' : 'bronze'},
    comment = "Raw orders data ingested from the source system"
)

def create_bronze_orders():
    return(
        spark.readStream
            .format("cloudFiles")
            .option("cloudFiles.format", "json")
            .option("coudFiles.inferColumnTypes", "true")
            .load("/Volumes/circuitbox/landing/operational_data/orders/")
            .select(
                "*",
                F.col("_metadata.file_path").alias("input_file_path"),
                F.col("current_timestamp").alias("ingestion_timestamp")
            )
    )




In [0]:
valid_records = {
    "valid_customer_id" : "customer_id IS NOT NULL",
    "valid_order_id" : "order_id IS NOT NULL"
}
@dlt.table(
    name = "silver_orders_clean",
    table_properties = {'quality' : 'silver'},
    comment = "Silver Orders Clean orders data"
)

@dlt.expect_all_or_fail(valid_records)
@dlt.expect("valid_order_status",
            "order_status IN ('Pending','Shipped','Cancelled','Completed')"
        )
@dlt.expect("valid_payment_method", 
            "payment_method IN ('Credit Card','Bank Transfer','Paypal')"
        )

def create_silver_orders_clean():
    return(
        spark.readStream
            .table("LIVE.bronze_orders")
            .select(
                "customer_id",
                "items",
                "order_id",
                "order_status",
                F.col("order_timestamp").cast("TIMESTAMP"),
                "payment_method"
            )

    )

In [0]:
dlt.create_streaming_table(
    name = "silver_orders",
    table_properties = {'quality' : 'silver'},
    comment = "Exploded silvere orders data"
)

In [0]:
items_schema = ArrayType(
    StructType([
        StructField("item_id", StringType(), True),
        StructField("name", StringType(), True),
        StructField("price", DoubleType(), True),
        StructField("quantity", IntegerType(), True),
        StructField("category", StringType(), True)
    ])
)

In [0]:
@dlt.view
def silver_orders_exploded():
    return(
        spark.readStream
            .table("silver_orders_clean")
            .withColumn(
                "items_array",
                F.from_json(
                    F.col("items"),
                    items_schema
                )
            )
            .withColumn(
                "item",
                F.explode("items_array")
                )
            .select(
                "order_id",
                "customer_id",
                "order_timestamp",
                "payment_method",
                "order_status",
                F.col("item.item_id").alias("item_id"),
                F.col("item.name").alias("item_name"),
                F.col("item.price").alias("item_price"),
                F.col("item.quantity").alias("item_quantity"),
                F.col("item.category").alias("item_category")
            )
        
    )

In [0]:
dlt.apply_changes(
    target = "silver_orders",
    source = "silver_orders_exploded",
    keys = ["order_id", "item_id"],
    sequence_by = "order_timestamp"
)